In [18]:

# PHASE 1 — Load dataset & create Target_NextGW + Split

import pandas as pd
import numpy as np

print("Loading dataset...")

df = pd.read_csv("../output/training_data_v6_no0min_trial.csv", encoding="utf-8-sig")
print(f"   → Loaded {len(df):,} rows")

# Sort dataset
df = df.sort_values(["Player UUID", "season", "Gameweek"]).reset_index(drop=True)

# Create Target_NextGW
df["Target_NextGW"] = df.groupby(["Player UUID", "season"])["Total Points"].shift(-1)

# Drop rows with no target
before = len(df)
df = df.dropna(subset=["Target_NextGW"]).reset_index(drop=True)
print(f"🧹 Removed rows with no next GW target: {before - len(df)}")

df["Target_NextGW"] = df["Target_NextGW"].astype(float)

print("✔ Target ready.")

# Build Split (last-5 global per player)
print("\n Building last-5-observations Validation split...")

df["Split"] = "Train"

for uuid, g in df.groupby("Player UUID", sort=False):
    last_5_idx = g.tail(5).index
    df.loc[last_5_idx, "Split"] = "Validation"

print(df["Split"].value_counts())
print("✔ Split complete.\n")


Loading dataset...
   → Loaded 25,735 rows
🧹 Removed rows with no next GW target: 2314
✔ Target ready.

 Building last-5-observations Validation split...
Split
Train         19942
Validation     3479
Name: count, dtype: int64
✔ Split complete.



In [19]:
print(" Checking dataframe...")

print("Rows:", len(df))
print("Columns:", list(df.columns))
print(df["Split"].value_counts())

print("✔ DataFrame OK.\n")


 Checking dataframe...
Rows: 23421
Columns: ['Player UUID', 'Code', 'Player Name', 'Web Name', 'Player Team Name', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Yellow Card', 'Red Cards', 'Total Points', 'Threat', 'ICT Index', 'Influence', 'Creativity', 'Opponent Name', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Goals Conceded_L3', 'Avg_ICT Index_L3', 'Avg_Threat_L3', 'Avg_Creativity_L3', 'Avg_Influence_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5', 'Avg_Goals Conceded_L5', 'Avg_ICT Index_L5', 'Avg_Threat_L5', 'Avg_Creativity_L5', 'Avg_Influence_L5', 'Team_Total_Points_GW', 'Team_Total_Points', 'Player_Season_Points', 'Team_Points_Contribution_GW', 'Team_Points_Contribution_Causal', 'Team_Points_Contribution_GW_Pct', 'Team_Points_Contribution_Causal_Pct', 'Team_Contrib

In [ ]:

# CELL 5 — Raw Feature Preparation for Tree Models (no scaling)

#remove leakage : Player UUID, season, Gameweek, Split, Opponent Name
def prepare_raw_features(df):

    df_model = df.copy()

    cols_to_drop = [
        "Player UUID", "Player Name", "Web Name",
        "Player Team Name", "Opponent Name",
        "season", "Gameweek", "Split"
    ]
    df_model = df_model.drop(columns=cols_to_drop, errors="ignore")

    y = df_model["Target_NextGW"].astype(float)
    X = df_model.drop(columns=["Target_NextGW"], errors="ignore")

    # Fix Is Home : Is Home --> 0/1

    if "Is Home" in X.columns:
        X["Is Home"] = X["Is Home"].replace(
            {True: 1, False: 0, "True": 1, "False": 0}
        ).astype(int)

    # One-hot Position 
    if "Position" in X.columns:
        X["Position"] = X["Position"].fillna("Unknown")
        X = pd.get_dummies(X, columns=["Position"], drop_first=True)

    # Fill missing
    X = X.fillna(X.median())

    train_mask = df["Split"] == "Train"
    val_mask   = df["Split"] == "Validation"

    X_train = X.loc[train_mask].reset_index(drop=True)
    X_val   = X.loc[val_mask].reset_index(drop=True)
    y_train    = y.loc[train_mask].reset_index(drop=True)
    y_val      = y.loc[val_mask].reset_index(drop=True)

    return X_train, X_val, y_train, y_val, X.columns


X_train, X_val, y_train, y_val, feature_names = prepare_raw_features(df)

print("✔ Raw features ready for tree models.")
print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)


✔ Raw features ready for tree models.
Train shape: (19942, 47)
Val shape: (3479, 47)


C:\Users\SOFI\AppData\Local\Temp\ipykernel_18580\549411864.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X["Is Home"] = X["Is Home"].replace(


## **LightGBM**

In [ ]:

# CELL 6 — LightGBM Optuna Objective 

import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

def objective_lgbm(trial):

    params = {
        "objective": "regression",
        "metric": "mae",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "random_state": 42,

        "num_leaves": trial.suggest_int("num_leaves", 20, 250),
        "max_depth": trial.suggest_int("max_depth", -1, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),

        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
    }

    model = LGBMRegressor(**params)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    return mae

print("✔ Corrected LightGBM objective function ready.")


✔ Corrected LightGBM objective function ready.


In [ ]:

# Run Optuna Study for LightGBM

study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=50)

print("LightGBM tuning completed.")
print("Best MAE:", study_lgbm.best_value)
print("Best parameters:", study_lgbm.best_params)



[I 2025-12-13 14:45:50,706] A new study created in memory with name: no-name-3c0d7d46-006e-4bcb-bb07-f2848cba45f3
[I 2025-12-13 14:45:51,854] Trial 0 finished with value: 1.7965555155666486 and parameters: {'num_leaves': 158, 'max_depth': 2, 'learning_rate': 0.12195240997824801, 'n_estimators': 1953, 'min_child_samples': 43, 'subsample': 0.6819392204234381, 'colsample_bytree': 0.5311714316639251, 'reg_alpha': 0.5437608377106702, 'reg_lambda': 0.8440720527258597}. Best is trial 0 with value: 1.7965555155666486.
[I 2025-12-13 14:45:52,594] Trial 1 finished with value: 1.780639660153401 and parameters: {'num_leaves': 141, 'max_depth': 2, 'learning_rate': 0.029191129369975626, 'n_estimators': 1118, 'min_child_samples': 38, 'subsample': 0.9804685459500095, 'colsample_bytree': 0.8862719338797567, 'reg_alpha': 0.8545166106664599, 'reg_lambda': 0.714078713583234}. Best is trial 1 with value: 1.780639660153401.
[I 2025-12-13 14:45:56,999] Trial 2 finished with value: 1.8569551056079736 and para

LightGBM tuning completed.
Best MAE: 1.763320417003546
Best parameters: {'num_leaves': 184, 'max_depth': 13, 'learning_rate': 0.011432734105498086, 'n_estimators': 958, 'min_child_samples': 81, 'subsample': 0.7613478674009443, 'colsample_bytree': 0.8949263542564202, 'reg_alpha': 0.2982270239297516, 'reg_lambda': 0.4706290936172891}


In [23]:

# CELL 7 — Train Final LightGBM Model with Best Parameters

print("Training final LightGBM model with best Optuna parameters...")

# Use the correct study variable name
best_params = study_lgbm.best_params.copy()

# Add LightGBM-required parameters
best_params.update({
    "objective": "regression",
    "metric": "mae",
    "random_state": 42,
    "verbosity": -1
})

final_lgbm = LGBMRegressor(**best_params)

final_lgbm.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="mae"
)

print("✔ Final LightGBM model trained.")
print("Best parameters used:", best_params)


Training final LightGBM model with best Optuna parameters...
✔ Final LightGBM model trained.
Best parameters used: {'num_leaves': 184, 'max_depth': 13, 'learning_rate': 0.011432734105498086, 'n_estimators': 958, 'min_child_samples': 81, 'subsample': 0.7613478674009443, 'colsample_bytree': 0.8949263542564202, 'reg_alpha': 0.2982270239297516, 'reg_lambda': 0.4706290936172891, 'objective': 'regression', 'metric': 'mae', 'random_state': 42, 'verbosity': -1}


In [ ]:

# CELL 8 — Predictions, Errors, Feature Importance & Save Results

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

print("Generating predictions and exporting outputs...")


df_model = df.copy()

cols_to_drop = [
    "Player UUID", "Player Name", "Web Name",
    "Player Team Name", "Opponent Name",
    "season", "Gameweek", "Split"
]
df_model = df_model.drop(columns=cols_to_drop, errors="ignore")

y_full = df_model["Target_NextGW"]
X_full = df_model.drop(columns=["Target_NextGW"], errors="ignore")

# Fix Is Home
if "Is Home" in X_full.columns:
    X_full["Is Home"] = X_full["Is Home"].replace(
        {True:1, False:0, "True":1, "False":0}
    ).astype(int)

# Position → One hot encode
if "Position" in X_full.columns:
    X_full["Position"] = X_full["Position"].fillna("Unknown")
    X_full = pd.get_dummies(X_full, columns=["Position"], drop_first=True)

# Fill missing values
X_full = X_full.fillna(X_full.median())

# Apply SAME scaler as train


# 1. Predict

train_pred = final_lgbm.predict(X_train)
val_pred   = final_lgbm.predict(X_val)
all_pred = final_lgbm.predict(X_full)


# 2. Performance

mae_train = mean_absolute_error(y_train, train_pred)
mae_val   = mean_absolute_error(y_val, val_pred)

rmse_train = mean_squared_error(y_train, train_pred) ** 0.5
rmse_val   = mean_squared_error(y_val, val_pred) ** 0.5

print("Performance:")
print(f"  MAE Train: {mae_train:.4f}")
print(f"  MAE Validation: {mae_val:.4f}")
print(f"  RMSE Train: {rmse_train:.4f}")
print(f"  RMSE Validation: {rmse_val:.4f}")


# 3. Prepare output directory

OUT_DIR = Path("../output/tree_models/lightgbm/")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# 4. Build merged predictions table
df_out = df.copy()
df_out["Prediction_For_GW"] = df_out["Gameweek"] + 1
df_out["Total_Points_Actual"] = df_out["Target_NextGW"]
df_out["Total_Points_Predicted"] = all_pred
df_out["Error"] = (df_out["Total_Points_Actual"] - df_out["Total_Points_Predicted"]).abs()

keep_cols = [
    "Player UUID", "Player Name", "Web Name",
    "season", "Prediction_For_GW", "Opponent Difficulty",
    "Total_Points_Actual", "Total_Points_Predicted",
    "Error", "Split"
]

df_final = df_out[keep_cols]

df_final.to_csv(OUT_DIR / "lgbm_predictions.csv", index=False, encoding="utf-8-sig")
df_final[df_final["Split"] == "Train"].to_csv(OUT_DIR / "lgbm_train.csv", index=False, encoding="utf-8-sig")
df_final[df_final["Split"] == "Validation"].to_csv(OUT_DIR / "lgbm_validation.csv", index=False, encoding="utf-8-sig")

print("✔ Predictions exported.")


# 5. Feature importance
importance_df = pd.DataFrame({
    "Feature": X_full.columns,
    "Importance": final_lgbm.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df.to_csv(OUT_DIR / "feature_importance.csv", index=False, encoding="utf-8-sig")

print("✔ Feature importance exported.")


# 6. Save model

final_lgbm.booster_.save_model(str(OUT_DIR / "lightgbm_model.txt"))
print("✔ LightGBM model saved.")

# -----------------------------------------------------------
# SUMMARY

print("\nLightGBM Export Complete")
print("---------------------------")
print(f"MAE Train:       {mae_train:.4f}")
print(f"MAE Validation:  {mae_val:.4f}")
print(f"RMSE Train:      {rmse_train:.4f}")
print(f"RMSE Validation: {rmse_val:.4f}")


Generating predictions and exporting outputs...


C:\Users\SOFI\AppData\Local\Temp\ipykernel_18580\989347832.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_full["Is Home"] = X_full["Is Home"].replace(


Performance:
  MAE Train: 1.6488
  MAE Validation: 1.7633
  RMSE Train: 2.3038
  RMSE Validation: 2.5304
✔ Predictions exported.
✔ Feature importance exported.
✔ LightGBM model saved.

LightGBM Export Complete
---------------------------
MAE Train:       1.6488
MAE Validation:  1.7633
RMSE Train:      2.3038
RMSE Validation: 2.5304


In [25]:

# CELL 9 — Update Model_Performance.csv for LightGBM

import pandas as pd
import numpy as np
from pathlib import Path

print("Updating Model_Performance.csv for LightGBM...")


# 1. Path setup

MODEL_PERF_PATH = Path("../output/model_performance/Model_Performance.csv")


# 2. Load existing performance table (or create new)

if MODEL_PERF_PATH.exists():
    perf = pd.read_csv(MODEL_PERF_PATH)
else:
    perf = pd.DataFrame(columns=[
        "Model", "MAE_Train", "MAE_Validation",
        "RMSE_Train", "RMSE_Validation",
        "Relative_Improvement_vs_Baseline"
    ])


# 3. Remove old LightGBM entries

perf = perf[~perf["Model"].str.contains("LightGBM", na=False)]


# 4. Find Baseline MAE (Rolling Average)

baseline_row = perf[perf["Model"] == "Rolling Average (TotalPoints)"]

if len(baseline_row) > 0:
    baseline_mae = baseline_row["MAE_Validation"].iloc[0]
else:
    baseline_mae = mae_val   # fallback if not found


# 5. Compute relative improvement

relative_impr = (baseline_mae - mae_val) / baseline_mae


# 6. Add new LightGBM row

new_row = pd.DataFrame([{
    "Model": "LightGBM (Optuna Tuned)",
    "MAE_Train": round(mae_train, 5),
    "MAE_Validation": round(mae_val, 5),
    "RMSE_Train": round(rmse_train, 5),
    "RMSE_Validation": round(rmse_val, 5),
    "Relative_Improvement_vs_Baseline": round(relative_impr, 5)
}])

perf = pd.concat([perf, new_row], ignore_index=True)

# 7. Save updated table

perf.to_csv(MODEL_PERF_PATH, index=False, encoding="utf-8-sig")

print("✔ Model_Performance.csv updated successfully.")

print("\nNew LightGBM entry:")
display(new_row)


Updating Model_Performance.csv for LightGBM...
✔ Model_Performance.csv updated successfully.

New LightGBM entry:


,Model,MAE_Train,MAE_Validation,RMSE_Train,RMSE_Validation,Relative_Improvement_vs_Baseline
0,LightGBM (Optuna Tuned),1.64883,1.76332,2.30378,2.53038,-0.00373


In [ ]:

# CELL — SHAP Analysis for LightGBM Model

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("Running SHAP interpretability analysis...")

# Directory for SHAP outputs
SHAP_DIR = Path("../output/tree_models/lightgbm/shap/")
SHAP_DIR.mkdir(parents=True, exist_ok=True)


# 1. SHAP Explainer

explainer = shap.TreeExplainer(final_lgbm)

# Important: Use a smaller sample for SHAP to avoid heavy computation
sample_size = min(5000, X_train.shape[0])  
sample_idx = np.random.choice(X_train.index, sample_size, replace=False)

X_sample = X_train.iloc[sample_idx]
print(f"Using SHAP sample size: {len(X_sample)}")

# Compute SHAP values
shap_values = explainer.shap_values(X_sample)


# 2. Save SHAP values to CSV

shap_df = pd.DataFrame(shap_values, columns=feature_names)
shap_df.to_csv(SHAP_DIR / "shap_values_sample.csv", index=False, encoding="utf-8-sig")

print("✔ SHAP values saved.")



# 3. Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / "shap_summary_plot.png", dpi=200, bbox_inches='tight')
plt.close()
print("✔ SHAP summary plot saved.")


# 4. Bar Plot (Importance)

plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / "shap_bar_plot.png", dpi=200, bbox_inches='tight')
plt.close()
print("✔ SHAP bar plot saved.")


mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_df = (
    pd.DataFrame({
        "Feature": feature_names, 
        "MeanAbsSHAP": mean_abs_shap
    })
    .sort_values("MeanAbsSHAP", ascending=False)
)

shap_df.to_csv(SHAP_DIR / "shap_values_summary.csv", index=False, encoding="utf-8-sig")
print("✔ SHAP numeric summary exported → shap_values_summary.csv")

print("SHAP Analysis Complete.")
print("Outputs saved to:", SHAP_DIR)



Running SHAP interpretability analysis...
Using SHAP sample size: 5000
✔ SHAP values saved.
✔ SHAP summary plot saved.
✔ SHAP bar plot saved.
✔ SHAP numeric summary exported → shap_values_summary.csv
SHAP Analysis Complete.
Outputs saved to: ..\output\tree_models\lightgbm\shap


## **Random Forest model**

In [ ]:

# CELL 10 — Fast Random Forest Optuna Objective

import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

print("Initializing FAST Optuna study for Random Forest...")

def objective_rf(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),  # small range
        "max_depth": trial.suggest_int("max_depth", 5, 15),           # no deep trees
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "bootstrap": True,
        "n_jobs": -1,
        "random_state": 42
    }

    model = RandomForestRegressor(**params)

    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    return mean_absolute_error(y_val, pred)

print("✔ FAST Random Forest objective ready.")


Initializing FAST Optuna study for Random Forest...
✔ FAST Random Forest objective ready.


In [ ]:

# CELL 11 — Run Fast Random Forest Search

study_rf = optuna.create_study(direction="minimize")
study_rf.optimize(objective_rf, n_trials=15)   # Trials = 15

print("\nRandom Forest tuning completed.")
print("Best MAE:", study_rf.best_value)
print("Best params:", study_rf.best_params)


[I 2025-12-13 14:49:01,628] A new study created in memory with name: no-name-c4d5c334-d269-48e3-9819-0ba870f566cc
[I 2025-12-13 14:49:03,605] Trial 0 finished with value: 1.792323766897275 and parameters: {'n_estimators': 225, 'max_depth': 15, 'min_samples_split': 14, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 0 with value: 1.792323766897275.
[I 2025-12-13 14:49:06,324] Trial 1 finished with value: 1.7956827199009648 and parameters: {'n_estimators': 290, 'max_depth': 14, 'min_samples_split': 11, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 0 with value: 1.792323766897275.
[I 2025-12-13 14:49:09,320] Trial 2 finished with value: 1.8101965759933007 and parameters: {'n_estimators': 563, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 1.792323766897275.
[I 2025-12-13 14:49:12,065] Trial 3 finished with value: 1.7994128616262162 and parameters: {'n_estimators': 323, 'max_depth': 12, 'min_


Random Forest tuning completed.
Best MAE: 1.7923222413369015
Best params: {'n_estimators': 417, 'max_depth': 12, 'min_samples_split': 15, 'min_samples_leaf': 8, 'max_features': 'sqrt'}


In [ ]:

# CELL 12 — Train Final Random Forest Model + Export Outputs

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import pandas as pd
import numpy as np

print("Training final Random Forest model with best Optuna parameters...")


# 1. Get best parameters from Optuna

best_rf_params = study_rf.best_params.copy()
best_rf_params.update({
    "n_jobs": -1,
    "random_state": 42
})

final_rf = RandomForestRegressor(**best_rf_params)

# Fit model
final_rf.fit(X_train, y_train)

print("✔ Final Random Forest model trained.")
print("Best parameters:")
print(best_rf_params)


# 2. Predict for Train / Validation / Full dataset

train_pred = final_rf.predict(X_train)
val_pred   = final_rf.predict(X_val)

# Rebuild FULL X exactly as in feature engineering
df_model = df.copy()

cols_to_drop = [
    "Player UUID", "Player Name", "Web Name",
    "Player Team Name", "Opponent Name",
    "season", "Gameweek", "Split"
]
df_model = df_model.drop(columns=cols_to_drop, errors="ignore")

X_full = df_model.drop(columns=["Target_NextGW"], errors="ignore")
y_full = df_model["Target_NextGW"]

# Fix Is Home
if "Is Home" in X_full.columns:
    X_full["Is Home"] = X_full["Is Home"].replace(
        {True:1, False:0, "True":1, "False":0}
    ).astype(int)

# One-hot encode Position
if "Position" in X_full.columns:
    X_full["Position"] = X_full["Position"].fillna("Unknown")
    X_full = pd.get_dummies(X_full, columns=["Position"], drop_first=True)

# Fill missing values
X_full = X_full.fillna(X_full.median())

# FULL predictions
all_pred = final_rf.predict(X_full)


# 3. Compute performance metrics

mae_train = mean_absolute_error(y_train, train_pred)
mae_val   = mean_absolute_error(y_val, val_pred)

rmse_train = mean_squared_error(y_train, train_pred) ** 0.5
rmse_val   = mean_squared_error(y_val, val_pred) ** 0.5

print("\nPerformance:")
print(f"  MAE Train:      {mae_train:.4f}")
print(f"  MAE Validation: {mae_val:.4f}")
print(f"  RMSE Train:     {rmse_train:.4f}")
print(f"  RMSE Validation:{rmse_val:.4f}")


# 4. Output directory

OUT_DIR = Path("../output/tree_models/random_forest/")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# 5. Build predictions table

df_out = df.copy()
df_out["Prediction_For_GW"] = df_out["Gameweek"] + 1
df_out["Total_Points_Actual"] = df_out["Target_NextGW"]
df_out["Total_Points_Predicted"] = all_pred
df_out["Error"] = (df_out["Total_Points_Actual"] - df_out["Total_Points_Predicted"]).abs()

keep_cols = [
    "Player UUID", "Player Name", "Web Name",
    "season", "Prediction_For_GW", "Opponent Difficulty",
    "Total_Points_Actual", "Total_Points_Predicted",
    "Error", "Split"
]

df_final = df_out[keep_cols]

df_final.to_csv(OUT_DIR / "rf_predictions.csv", index=False, encoding="utf-8-sig")
df_final[df_final["Split"] == "Train"].to_csv(OUT_DIR / "rf_train.csv", index=False, encoding="utf-8-sig")
df_final[df_final["Split"] == "Validation"].to_csv(OUT_DIR / "rf_validation.csv", index=False, encoding="utf-8-sig")

print("Predictions exported.")


# 6. Feature importance export

feature_importance_df = pd.DataFrame({
    "Feature": X_full.columns,
    "Importance": final_rf.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance_df.to_csv(OUT_DIR / "feature_importance.csv", index=False, encoding="utf-8-sig")

print("Feature importance exported.")


# 7. Save model

import joblib
joblib.dump(final_rf, OUT_DIR / "random_forest_model.pkl")

print("Random Forest model saved.")


# Summary

print("\nRandom Forest Export Complete")
print("-----------------------------------")
print(f"MAE Train:       {mae_train:.4f}")
print(f"MAE Validation:  {mae_val:.4f}")
print(f"RMSE Train:      {rmse_train:.4f}")
print(f"RMSE Validation: {rmse_val:.4f}")


Training final Random Forest model with best Optuna parameters...
✔ Final Random Forest model trained.
Best parameters:
{'n_estimators': 417, 'max_depth': 12, 'min_samples_split': 15, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'n_jobs': -1, 'random_state': 42}


C:\Users\SOFI\AppData\Local\Temp\ipykernel_18580\1421826872.py:54: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_full["Is Home"] = X_full["Is Home"].replace(



Performance:
  MAE Train:      1.8617
  MAE Validation: 1.7923
  RMSE Train:     2.5721
  RMSE Validation:2.5267
Predictions exported.
Feature importance exported.
Random Forest model saved.

Random Forest Export Complete
-----------------------------------
MAE Train:       1.8617
MAE Validation:  1.7923
RMSE Train:      2.5721
RMSE Validation: 2.5267


In [ ]:

# CELL 13 — Update Model_Performance.csv for Random Forest

import pandas as pd
import numpy as np
from pathlib import Path

print("Updating Model_Performance.csv for Random Forest...")

MODEL_PERF_PATH = Path("../output/model_performance/Model_Performance.csv")


# 1. Load existing performance table

if MODEL_PERF_PATH.exists():
    perf = pd.read_csv(MODEL_PERF_PATH)
else:
    perf = pd.DataFrame(columns=[
        "Model", "MAE_Train", "MAE_Validation",
        "RMSE_Train", "RMSE_Validation",
        "Relative_Improvement_vs_Baseline"
    ])


# 2. Remove previous RF entries to avoid duplicates
perf = perf[~perf["Model"].str.contains("Random Forest", na=False)]


# 3. Extract baseline MAE (Rolling Average)
baseline_row = perf[perf["Model"] == "Rolling Average (TotalPoints)"]

if len(baseline_row) > 0:
    baseline_mae = baseline_row["MAE_Validation"].iloc[0]
else:
    baseline_mae = mae_val  # fallback (should not happen ideally)


# 4. Compute Relative Improvement
relative_impr = (baseline_mae - mae_val) / baseline_mae


# 5. Create new entry for Random Forest
new_row = pd.DataFrame([{
    "Model": "Random Forest (Optuna Tuned)",
    "MAE_Train": round(mae_train, 5),
    "MAE_Validation": round(mae_val, 5),
    "RMSE_Train": round(rmse_train, 5),
    "RMSE_Validation": round(rmse_val, 5),
    "Relative_Improvement_vs_Baseline": round(relative_impr, 5)
}])

# Append to table
perf = pd.concat([perf, new_row], ignore_index=True)

# -------------------------------------------------------------
# 6. Save updated performance file
perf.to_csv(MODEL_PERF_PATH, index=False, encoding="utf-8-sig")

print("✔ Model_Performance.csv updated successfully.")
print("\nNew Random Forest entry:")
display(new_row)


Updating Model_Performance.csv for Random Forest...
✔ Model_Performance.csv updated successfully.

New Random Forest entry:


,Model,MAE_Train,MAE_Validation,RMSE_Train,RMSE_Validation,Relative_Improvement_vs_Baseline
0,Random Forest (Optuna Tuned),1.8617,1.79232,2.57212,2.52669,-0.02024


In [ ]:

# SHAP Analysis for Random Forest

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("Running SHAP analysis for Random Forest...")

# Output directory
SHAP_DIR = Path("../output/tree_models/random_forest/shap/")
SHAP_DIR.mkdir(parents=True, exist_ok=True)


# 1. SHAP Explainer (TreeExplainer for RF)
explainer_rf = shap.TreeExplainer(final_rf)

# Use smaller sample because RF SHAP is heavy
sample_size = min(2000, X_train.shape[0])
sample_idx = np.random.choice(X_train.index, sample_size, replace=False)
X_sample_rf = X_train.iloc[sample_idx]

print(f"Using SHAP sample size for RF: {len(X_sample_rf)}")

# Compute SHAP values
shap_values_rf = explainer_rf.shap_values(X_sample_rf)


# 2. Save SHAP values

shap_rf_df = pd.DataFrame(shap_values_rf, columns=feature_names)
shap_rf_df.to_csv(SHAP_DIR / "shap_values_sample_rf.csv", index=False, encoding="utf-8-sig")

print("✔ SHAP values for RF saved.")


# 3. SHAP Summary Plot

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_rf, X_sample_rf, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / "shap_summary_plot_rf.png", dpi=200, bbox_inches='tight')
plt.close()
print("✔ RF SHAP summary plot saved.")


# 4. SHAP Bar Plot

plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values_rf, X_sample_rf, feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / "shap_bar_plot_rf.png", dpi=200, bbox_inches='tight')
plt.close()
print("✔ RF SHAP bar plot saved.")

mean_abs_shap = np.abs(shap_values_rf).mean(axis=0)

shap_df = (
    pd.DataFrame({
        "Feature": X_val.columns,
        "MeanAbsSHAP": mean_abs_shap
    })
    .sort_values("MeanAbsSHAP", ascending=False)
)

shap_df.to_csv(SHAP_DIR / "shap_values_summary.csv", index=False, encoding="utf-8-sig")
print("✔ SHAP numeric summary exported → shap_values_summary.csv")

print("SHAP for Random Forest complete.")
print("Outputs saved to:", SHAP_DIR)


Running SHAP analysis for Random Forest...
Using SHAP sample size for RF: 2000
✔ SHAP values for RF saved.
✔ RF SHAP summary plot saved.
✔ RF SHAP bar plot saved.
✔ SHAP numeric summary exported → shap_values_summary.csv
SHAP for Random Forest complete.
Outputs saved to: ..\output\tree_models\random_forest\shap


## **GBM model**


In [ ]:

# CELL 14 — GBM Optuna Objective

import optuna
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

print("Initializing GBM Optuna objective...")

def objective_gbm(trial):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 600),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
    }

    model = GradientBoostingRegressor(**params)

    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    return mae

print("✔ GBM Optuna objective ready.")


Initializing GBM Optuna objective...
✔ GBM Optuna objective ready.


In [ ]:

# CELL 15 — Run GBM Optuna Tuning

print("Running GBM Optuna tuning...")

study_gbm = optuna.create_study(direction="minimize")
study_gbm.optimize(objective_gbm, n_trials=20)

print("\nGBM tuning completed.")
print("Best MAE:", study_gbm.best_value)
print("Best parameters:", study_gbm.best_params)


[I 2025-12-13 14:52:21,859] A new study created in memory with name: no-name-b53d026f-ad55-4823-bcb2-a22abfda5b81


Running GBM Optuna tuning...


[I 2025-12-13 14:52:46,324] Trial 0 finished with value: 1.778474798283515 and parameters: {'learning_rate': 0.05590395919299444, 'n_estimators': 280, 'max_depth': 4, 'min_samples_split': 25, 'min_samples_leaf': 15, 'subsample': 0.8548590083635217, 'max_features': None}. Best is trial 0 with value: 1.778474798283515.
[I 2025-12-13 14:52:50,578] Trial 1 finished with value: 1.8034287638000719 and parameters: {'learning_rate': 0.010743856947614134, 'n_estimators': 343, 'max_depth': 4, 'min_samples_split': 26, 'min_samples_leaf': 5, 'subsample': 0.7200472178273912, 'max_features': 'sqrt'}. Best is trial 0 with value: 1.778474798283515.
[I 2025-12-13 14:52:53,663] Trial 2 finished with value: 1.8093860543639417 and parameters: {'learning_rate': 0.012810906359606652, 'n_estimators': 433, 'max_depth': 2, 'min_samples_split': 47, 'min_samples_leaf': 14, 'subsample': 0.7938132020869216, 'max_features': 'sqrt'}. Best is trial 0 with value: 1.778474798283515.
[I 2025-12-13 14:52:59,756] Trial 3 


GBM tuning completed.
Best MAE: 1.7710901381940716
Best parameters: {'learning_rate': 0.07147659774508328, 'n_estimators': 264, 'max_depth': 4, 'min_samples_split': 24, 'min_samples_leaf': 20, 'subsample': 0.7902795813139447, 'max_features': None}


In [ ]:

# CELL 16 — Train Final GBM Model with Best Parameters & Export Results

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import pandas as pd
import numpy as np

print("Training final GBM model with best Optuna parameters...")


# 1. Get best parameters

best_params_gbm = study_gbm.best_params.copy()

# Add mandatory sklearn parameters
best_params_gbm.update({
    "loss": "squared_error",
    "random_state": 42
})

final_gbm = GradientBoostingRegressor(**best_params_gbm)

# Train final model
final_gbm.fit(X_train, y_train)

print("✔ Final GBM model trained.")
print("Best parameters used:", best_params_gbm)



# 2. Predict on train/validation/full dataset

train_pred = final_gbm.predict(X_train)
val_pred   = final_gbm.predict(X_val)

# Rebuild full X the same way as in raw-data preparation
df_model = df.copy()

cols_to_drop = [
    "Player UUID", "Player Name", "Web Name",
    "Player Team Name", "Opponent Name",
    "season", "Gameweek", "Split"
]
df_model = df_model.drop(columns=cols_to_drop, errors="ignore")

y_full = df_model["Target_NextGW"]
X_full = df_model.drop(columns=["Target_NextGW"], errors="ignore")

# Fix Is Home
if "Is Home" in X_full.columns:
    X_full["Is Home"] = X_full["Is Home"].replace(
        {True:1, False:0, "True":1, "False":0}
    ).astype(int)

# One-hot Position
if "Position" in X_full.columns:
    X_full["Position"] = X_full["Position"].fillna("Unknown")
    X_full = pd.get_dummies(X_full, columns=["Position"], drop_first=True)

# Fill missing
X_full = X_full.fillna(X_full.median())

all_pred = final_gbm.predict(X_full)


# 3. Performance metrics

mae_train = mean_absolute_error(y_train, train_pred)
mae_val   = mean_absolute_error(y_val, val_pred)

rmse_train = mean_squared_error(y_train, train_pred) ** 0.5
rmse_val   = mean_squared_error(y_val, val_pred) ** 0.5

print("\nPerformance:")
print(f"  MAE Train:      {mae_train:.4f}")
print(f"  MAE Validation: {mae_val:.4f}")
print(f"  RMSE Train:     {rmse_train:.4f}")
print(f"  RMSE Validation:{rmse_val:.4f}")



# 4. Export predictions

OUT_DIR = Path("../output/tree_models/gbm/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_out = df.copy()
df_out["Prediction_For_GW"] = df_out["Gameweek"] + 1
df_out["Total_Points_Actual"] = df_out["Target_NextGW"]
df_out["Total_Points_Predicted"] = all_pred
df_out["Error"] = (df_out["Total_Points_Actual"] - df_out["Total_Points_Predicted"]).abs()

keep_cols = [
    "Player UUID", "Player Name", "Web Name",
    "season", "Prediction_For_GW", "Opponent Difficulty",
    "Total_Points_Actual", "Total_Points_Predicted",
    "Error", "Split"
]

df_final = df_out[keep_cols]

df_final.to_csv(OUT_DIR / "gbm_predictions.csv", index=False, encoding="utf-8-sig")
df_final[df_final["Split"] == "Train"].to_csv(OUT_DIR / "gbm_train.csv", index=False)
df_final[df_final["Split"] == "Validation"].to_csv(OUT_DIR / "gbm_validation.csv", index=False)

print("✔ GBM predictions exported.")



# 5. Feature importance export

importance_df = pd.DataFrame({
    "Feature": X_full.columns,
    "Importance": final_gbm.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df.to_csv(OUT_DIR / "feature_importance.csv", index=False)
print("✔ Feature importance exported.")

# --------------------------------------------------------
# Summary
# --------------------------------------------------------

print("\nGBM Export Complete")
print("--------------------------------")
print(f"MAE Train:       {mae_train:.4f}")
print(f"MAE Validation:  {mae_val:.4f}")
print(f"RMSE Train:      {rmse_train:.4f}")
print(f"RMSE Validation: {rmse_val:.4f}")


Training final GBM model with best Optuna parameters...
✔ Final GBM model trained.
Best parameters used: {'learning_rate': 0.07147659774508328, 'n_estimators': 264, 'max_depth': 4, 'min_samples_split': 24, 'min_samples_leaf': 20, 'subsample': 0.7902795813139447, 'max_features': None, 'loss': 'squared_error', 'random_state': 42}

Performance:
  MAE Train:      1.9342
  MAE Validation: 1.7711
  RMSE Train:     2.6785
  RMSE Validation:2.5221


C:\Users\SOFI\AppData\Local\Temp\ipykernel_18580\877966184.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_full["Is Home"] = X_full["Is Home"].replace(


✔ GBM predictions exported.
✔ Feature importance exported.

GBM Export Complete
--------------------------------
MAE Train:       1.9342
MAE Validation:  1.7711
RMSE Train:      2.6785
RMSE Validation: 2.5221


In [ ]:

# CELL 17 — Update Model_Performance.csv for GBM

import pandas as pd
from pathlib import Path

print("Updating Model_Performance.csv for GBM...")

MODEL_PERF_PATH = Path("../output/model_performance/Model_Performance.csv")

# Load if exists
if MODEL_PERF_PATH.exists():
    perf = pd.read_csv(MODEL_PERF_PATH)
else:
    perf = pd.DataFrame(columns=[
        "Model", "MAE_Train", "MAE_Validation",
        "RMSE_Train", "RMSE_Validation",
        "Relative_Improvement_vs_Baseline"
    ])

# Remove old GBM rows
perf = perf[~perf["Model"].str.contains("GBM", na=False)]

# Baseline from Rolling Average
baseline_row = perf[perf["Model"] == "Rolling Average (TotalPoints)"]
baseline_mae = baseline_row["MAE_Validation"].iloc[0] if len(baseline_row) > 0 else mae_val

# Relative improvement
relative_impr = (baseline_mae - mae_val) / baseline_mae

new_row = pd.DataFrame([{
    "Model": "GBM (Optuna Tuned)",
    "MAE_Train": round(mae_train, 5),
    "MAE_Validation": round(mae_val, 5),
    "RMSE_Train": round(rmse_train, 5),
    "RMSE_Validation": round(rmse_val, 5),
    "Relative_Improvement_vs_Baseline": round(relative_impr, 5)
}])

perf = pd.concat([perf, new_row], ignore_index=True)
perf.to_csv(MODEL_PERF_PATH, index=False, encoding="utf-8-sig")

print("✔ Model_Performance.csv updated.")
display(new_row)


Updating Model_Performance.csv for GBM...
✔ Model_Performance.csv updated.


,Model,MAE_Train,MAE_Validation,RMSE_Train,RMSE_Validation,Relative_Improvement_vs_Baseline
0,GBM (Optuna Tuned),1.93417,1.77109,2.67851,2.5221,-0.00815


In [ ]:

# CELL 18 — SHAP Explainability for GBM

import shap
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

print("Initializing SHAP for GBM...")

# Create SHAP output directory
GBM_DIR = Path("../output/tree_models/gbm/shap")
GBM_DIR.mkdir(parents=True, exist_ok=True)

# SHAP explainer for GradientBoostingRegressor
explainer = shap.TreeExplainer(final_gbm)

# Compute SHAP values ONLY on validation set (correct approach)
print("Computing SHAP values on validation set...")
shap_values = explainer.shap_values(X_val)

print("✔ SHAP values computed.")


# SUMMARY PLOT 

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_val, feature_names=X_val.columns, show=False)
plt.tight_layout()
plt.savefig(GBM_DIR / "shap_summary_plot.png", dpi=200, bbox_inches='tight')
plt.close()
print("✔ SHAP summary plot saved.")


# BAR PLOT (Feature importance by mean |SHAP|)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_val, feature_names=X_val.columns, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(GBM_DIR / "shap_feature_importance.png", dpi=200, bbox_inches='tight')
plt.close()
print("✔ SHAP bar plot saved.")


# EXPORT RAW SHAP TABLE

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_df = (
    pd.DataFrame({
        "Feature": X_val.columns,
        "MeanAbsSHAP": mean_abs_shap
    })
    .sort_values("MeanAbsSHAP", ascending=False)
)

shap_df.to_csv(GBM_DIR / "shap_values_summary.csv", index=False, encoding="utf-8-sig")
print("✔ SHAP numeric summary exported → shap_values_summary.csv")

print("\nSHAP Analysis Complete for GBM")


Initializing SHAP for GBM...
Computing SHAP values on validation set...
✔ SHAP values computed.
✔ SHAP summary plot saved.
✔ SHAP bar plot saved.
✔ SHAP numeric summary exported → shap_values_summary.csv

SHAP Analysis Complete for GBM
